In [3]:
# 1) Instalar dependencias
!pip install -q streamlit pyngrok

In [4]:
# 2) Escribir la app de Streamlit en el entorno de Colab
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Monitor de Desigualdad EPH - Corrientes (Grupo 5)",
    layout="wide",
    initial_sidebar_state="expanded"
)

COLUMNAS_MINIMAS = ['CODUSU', 'NRO_HOGAR', 'IPCF', 'PONDIH']
COLUMNAS_COMPOSICION = ['P21', 'TOT_P12', 'V2_M', 'V3_M', 'V4_M', 'V5_M']


def validar_columnas(df, nombre_base="la base cargada"):
    faltantes = [c for c in COLUMNAS_MINIMAS if c not in df.columns]
    if faltantes:
        st.error(f"⚠️ {nombre_base} no contiene las columnas requeridas: {', '.join(faltantes)}. "
                 f"Verificá que sea la Base Individual de la EPH (INDEC) sin modificar encabezados.")
        return False, faltantes
    return True, []


def procesar_base_eph(df_individual, df_hogar=None, aglomerado=12):
    df = df_individual.copy()
    if df_hogar is not None and {'CODUSU', 'NRO_HOGAR'}.issubset(df_hogar.columns):
        try:
            for col in ['CODUSU', 'NRO_HOGAR']:
                df[col] = df[col].astype(str)
                df_hogar[col] = df_hogar[col].astype(str)
            cols_hogar = [c for c in df_hogar.columns if c not in df.columns or c in ['CODUSU', 'NRO_HOGAR']]
            df = pd.merge(df, df_hogar[cols_hogar], on=['CODUSU', 'NRO_HOGAR'], how='inner')
        except Exception as e:
            st.warning(f"No se pudo vincular la Base Hogar automáticamente: {e}. Se continúa solo con Individual.")

    col_decil_nac = 'DECCFR' if 'DECCFR' in df.columns else ('P_DECCF' if 'P_DECCF' in df.columns else None)
    if col_decil_nac:
        df = df[~df[col_decil_nac].isin([0, 12, '0', '12'])]

    df = df[(df['IPCF'] > 0) & (df['PONDIH'] > 0)].copy()

    if aglomerado != 0 and 'AGLOMERADO' in df.columns:
        df = df[df['AGLOMERADO'] == aglomerado].copy()

    return df


def calcular_gini_trapecio(df, col_ingreso='IPCF', col_ponderador='PONDIH'):
    if df.empty:
        return 0.0, pd.DataFrame({'X_poblacion': [0, 1], 'Y_ingreso': [0, 1]})

    df_sorted = df.sort_values(by=col_ingreso).copy()
    poblacion_ponderada = df_sorted[col_ponderador]
    ingreso_ponderado = df_sorted[col_ingreso] * df_sorted[col_ponderador]

    X = np.cumsum(poblacion_ponderada) / np.sum(poblacion_ponderada)
    Y = np.cumsum(ingreso_ponderado) / np.sum(ingreso_ponderado)
    X = np.insert(X.values, 0, 0.0)
    Y = np.insert(Y.values, 0, 0.0)

    dX = np.diff(X)
    Y_sum = Y[1:] + Y[:-1]
    area_bajo_lorenz = np.sum(dX * Y_sum) / 2.0
    coeficiente_gini = 1.0 - (2.0 * area_bajo_lorenz)

    df_lorenz = pd.DataFrame({'X_poblacion': X, 'Y_ingreso': Y})
    return coeficiente_gini, df_lorenz


def asignar_decil_local(df, col_ingreso='IPCF', col_ponderador='PONDIH'):
    if df.empty:
        return df.copy()

    df_sorted = df.sort_values(by=col_ingreso).copy()
    df_sorted['pob_acum'] = np.cumsum(df_sorted[col_ponderador])
    pob_total = df_sorted[col_ponderador].sum()

    df_sorted['decil_local'] = pd.cut(
        df_sorted['pob_acum'] / pob_total,
        bins=np.linspace(0, 1, 11),
        labels=range(1, 11),
        include_lowest=True
    )
    return df_sorted


def calcular_deciles_locales(df_con_decil, col_ingreso='IPCF', col_ponderador='PONDIH'):
    if df_con_decil.empty or 'decil_local' not in df_con_decil.columns:
        return pd.DataFrame()

    df_sorted = df_con_decil.copy()
    df_sorted['ingreso_total_reg'] = df_sorted[col_ingreso] * df_sorted[col_ponderador]

    resumen_deciles = df_sorted.groupby('decil_local', observed=False).agg(
        poblacion=(col_ponderador, 'sum'),
        masa_ingreso=('ingreso_total_reg', 'sum'),
        ipcf_promedio=(col_ingreso, lambda x: np.average(x, weights=df_sorted.loc[x.index, col_ponderador]))
    ).reset_index()

    resumen_deciles['pct_ingreso'] = (resumen_deciles['masa_ingreso'] / df_sorted['ingreso_total_reg'].sum()) * 100
    return resumen_deciles


def calcular_composicion_ingreso(df_con_decil, col_ponderador='PONDIH'):
    if df_con_decil.empty or 'decil_local' not in df_con_decil.columns:
        return pd.DataFrame(), []

    df = df_con_decil.copy()
    columnas_faltantes = [c for c in COLUMNAS_COMPOSICION if c not in df.columns]
    for c in COLUMNAS_COMPOSICION:
        if c not in df.columns:
            df[c] = 0.0
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)

    df['ing_laboral'] = df['P21'].clip(lower=0) + df['TOT_P12'].clip(lower=0)
    df['ing_jubilacion'] = df['V2_M'].clip(lower=0)
    df['ing_indemnizacion'] = df['V3_M'].clip(lower=0)
    df['ing_subsidios'] = df['V4_M'].clip(lower=0) + df['V5_M'].clip(lower=0)
    df['ing_no_laboral'] = df['ing_jubilacion'] + df['ing_indemnizacion'] + df['ing_subsidios']

    for col in ['ing_laboral', 'ing_jubilacion', 'ing_indemnizacion', 'ing_subsidios', 'ing_no_laboral']:
        df[f'{col}_pond'] = df[col] * df[col_ponderador]

    resumen = df.groupby('decil_local', observed=False).agg(
        laboral=('ing_laboral_pond', 'sum'),
        jubilacion=('ing_jubilacion_pond', 'sum'),
        indemnizacion=('ing_indemnizacion_pond', 'sum'),
        subsidios=('ing_subsidios_pond', 'sum'),
        no_laboral=('ing_no_laboral_pond', 'sum'),
    ).reset_index()

    resumen['total'] = resumen['laboral'] + resumen['no_laboral']
    for col in ['laboral', 'jubilacion', 'indemnizacion', 'subsidios', 'no_laboral']:
        resumen[f'{col}_pct'] = np.where(resumen['total'] > 0, (resumen[col] / resumen['total']) * 100, 0.0)

    return resumen, columnas_faltantes


def calcular_comparacion_decil_nacional_local(df_con_decil):
    col_decil_nac = None
    if 'DECCFR' in df_con_decil.columns:
        col_decil_nac = 'DECCFR'
    elif 'P_DECCF' in df_con_decil.columns:
        col_decil_nac = 'P_DECCF'

    if col_decil_nac is None or df_con_decil.empty or 'decil_local' not in df_con_decil.columns:
        return pd.DataFrame(), col_decil_nac

    df = df_con_decil.copy()
    df[col_decil_nac] = pd.to_numeric(df[col_decil_nac], errors='coerce')
    df = df.dropna(subset=[col_decil_nac])

    tabla = df.groupby('decil_local', observed=False).agg(
        decil_nacional_promedio=(col_decil_nac, 'mean'),
        n_casos=(col_decil_nac, 'count')
    ).reset_index()
    tabla['brecha_local_vs_nacional'] = tabla['decil_local'].astype(float) - tabla['decil_nacional_promedio']
    return tabla, col_decil_nac


def generar_datos_simulados_eph():
    np.random.seed(42)
    n = 1200

    def _simular_periodo(mean_log, sigma_log, ponb_min, ponb_max):
        ipcf = np.random.lognormal(mean=mean_log, sigma=sigma_log, size=n)
        ipcf_rank = pd.Series(ipcf).rank(pct=True).values
        prop_laboral = np.clip(0.35 + 0.55 * ipcf_rank + np.random.normal(0, 0.08, n), 0.05, 0.98)

        ingreso_total_hogar = ipcf * np.random.randint(1, 5, size=n)
        ing_laboral = ingreso_total_hogar * prop_laboral
        resto = ingreso_total_hogar - ing_laboral

        w_jub = np.random.dirichlet([2, 0.3, 1.2], size=n)
        v2 = resto * w_jub[:, 0]
        v3 = resto * w_jub[:, 1]
        v4v5 = resto * w_jub[:, 2]

        return pd.DataFrame({
            'CODUSU': [f'USU_{i}' for i in range(n)],
            'NRO_HOGAR': 1,
            'AGLOMERADO': 12,
            'IPCF': ipcf,
            'PONDIH': np.random.randint(ponb_min, ponb_max, size=n),
            'DECCFR': np.random.randint(1, 11, size=n),
            'P21': ing_laboral * 0.8,
            'TOT_P12': ing_laboral * 0.2,
            'V2_M': v2,
            'V3_M': v3,
            'V4_M': v4v5 * 0.7,
            'V5_M': v4v5 * 0.3,
        })

    df_2024 = _simular_periodo(11.5, 0.75, 150, 450)
    df_2025 = _simular_periodo(12.1, 0.82, 155, 460)
    return df_2024, df_2025


st.title("📊 Monitor de Distribución del Ingreso y Coeficiente de Gini (EPH)")
st.caption("Proyecto Final - Grupo 5: Andrea Celeste Coronel & Gisela Alejandra Romero | Tecnicatura en Ciencia de Datos e IA")

with st.expander("ℹ️ Nota metodológica (para el informe)"):
    st.markdown("""
    **Coeficiente de Gini:** método geométrico de los trapecios sobre la Curva de Lorenz,
    construida con `IPCF` ordenado ascendente y ponderado por `PONDIH`.
    `Gini = 1 - 2 * Área bajo la Curva de Lorenz`.

    **Deciles locales:** dividen la población ponderada del aglomerado en 10 tramos iguales.

    **Composición del ingreso:** laboral (`P21`+`TOT_P12`), jubilaciones (`V2_M`),
    indemnizaciones (`V3_M`) y subsidios/ayuda social (`V4_M`+`V5_M`), por decil local.

    **Ratio de Palma:** ingreso del Decil 10 / suma de ingresos de los Deciles 1 a 4.
    """)

st.sidebar.header("⚙️ Configuración y Carga de Datos")
fuente_datos = st.sidebar.radio("Fuente de Datos:", ["Muestra Oficial EPH (Corrientes)", "Cargar Archivos Excel/CSV"])

df_t424, df_t425 = None, None
datos_validos = True

if fuente_datos == "Cargar Archivos Excel/CSV":
    file_2024 = st.sidebar.file_uploader("Base Individual T4 2024", type=["xlsx", "csv"])
    file_2025 = st.sidebar.file_uploader("Base Individual T4 2025", type=["xlsx", "csv"])

    if file_2024 and file_2025:
        try:
            df_t424 = pd.read_excel(file_2024) if file_2024.name.endswith('.xlsx') else pd.read_csv(file_2024)
            df_t425 = pd.read_excel(file_2025) if file_2025.name.endswith('.xlsx') else pd.read_csv(file_2025)
        except Exception as e:
            st.sidebar.error(f"No se pudieron leer los archivos: {e}")
            datos_validos = False

        if datos_validos:
            ok_24, _ = validar_columnas(df_t424, "La Base T4 2024")
            ok_25, _ = validar_columnas(df_t425, "La Base T4 2025")
            datos_validos = ok_24 and ok_25

        if not datos_validos:
            st.stop()
    elif file_2024 or file_2025:
        st.sidebar.warning("Falta cargar el otro período. Usando la muestra simulada mientras tanto.")
        df_t424, df_t425 = generar_datos_simulados_eph()
    else:
        st.sidebar.info("Cargue ambos períodos (T4 2024 y T4 2025) para habilitar la auditoría completa.")
        df_t424, df_t425 = generar_datos_simulados_eph()
else:
    df_t424, df_t425 = generar_datos_simulados_eph()

aglom_opcion = st.sidebar.selectbox("Aglomerado de Análisis:", ["12 - Corrientes Capital", "Total Nacional (Todos los Aglomerados)"])
cod_aglomerado = 12 if "12" in aglom_opcion else 0

df_24_proc = procesar_base_eph(df_t424, aglomerado=cod_aglomerado)
df_25_proc = procesar_base_eph(df_t425, aglomerado=cod_aglomerado)

if df_24_proc.empty or df_25_proc.empty:
    st.error("Después de aplicar los filtros no quedaron registros válidos. Revisá el aglomerado seleccionado o la base cargada.")
    st.stop()

df_24_decil = asignar_decil_local(df_24_proc)
df_25_decil = asignar_decil_local(df_25_proc)

gini_24, lorenz_24 = calcular_gini_trapecio(df_24_proc)
gini_25, lorenz_25 = calcular_gini_trapecio(df_25_proc)

deciles_24 = calcular_deciles_locales(df_24_decil)
deciles_25 = calcular_deciles_locales(df_25_decil)

composicion_24, faltantes_comp_24 = calcular_composicion_ingreso(df_24_decil)
composicion_25, faltantes_comp_25 = calcular_composicion_ingreso(df_25_decil)

comparacion_nac_local_25, col_decil_nac = calcular_comparacion_decil_nacional_local(df_25_decil)

col1, col2, col3, col4 = st.columns(4)
col1.metric("Gini Corrientes T4 2024", f"{gini_24:.4f}")
diff_gini = gini_25 - gini_24
col2.metric("Gini Corrientes T4 2025", f"{gini_25:.4f}", delta=f"{diff_gini:+.4f}", delta_color="inverse")

pob_total_25 = int(df_25_proc['PONDIH'].sum())
col3.metric("Población Representada", f"{pob_total_25:,} hab.")

palma_ratio = None
if not deciles_25.empty:
    d10 = deciles_25.loc[deciles_25['decil_local'] == 10, 'pct_ingreso'].values[0]
    d1_4 = deciles_25.loc[deciles_25['decil_local'].isin([1, 2, 3, 4]), 'pct_ingreso'].sum()
    palma_ratio = d10 / d1_4 if d1_4 > 0 else 0
    col4.metric("Ratio de Palma (T4 2025)", f"{palma_ratio:.2f}x")

st.markdown("---")

tab1, tab2, tab3, tab4 = st.tabs([
    "📉 Curva de Lorenz (Trapecios)",
    "📊 Deciles Locales (2024 vs 2025) y Comparación Nacional",
    "🔍 Indicadores de Polarización",
    "🧩 Composición del Ingreso (Hogares Vulnerables)"
])

with tab1:
    st.subheader("Curva de Lorenz y Coeficiente de Gini (Método Trapezoidal)")

    fig_lorenz = go.Figure()
    fig_lorenz.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Equidad Perfecta (45°)', line=dict(dash='dash', color='gray')))

    step_24 = max(1, len(lorenz_24) // 200)
    step_25 = max(1, len(lorenz_25) // 200)

    fig_lorenz.add_trace(go.Scatter(x=lorenz_24['X_poblacion'].iloc[::step_24], y=lorenz_24['Y_ingreso'].iloc[::step_24], mode='lines', name=f'Corrientes T4 2024 (Gini: {gini_24:.4f})', line=dict(color='blue')))
    fig_lorenz.add_trace(go.Scatter(x=lorenz_25['X_poblacion'].iloc[::step_25], y=lorenz_25['Y_ingreso'].iloc[::step_25], mode='lines', name=f'Corrientes T4 2025 (Gini: {gini_25:.4f})', line=dict(color='red')))

    fig_lorenz.update_layout(
        title="Curva de Lorenz sobre Ingreso Per Cápita Familiar (IPCF)",
        xaxis_title="Proporción Acumulada de la Población (X)",
        yaxis_title="Proporción Acumulada del Ingreso (Y)",
        template="plotly_white",
        height=500
    )
    st.plotly_chart(fig_lorenz, use_container_width=True)

with tab2:
    st.subheader("Distribución de la Masa de Ingresos por Decil Local")

    df_deciles_comp = pd.merge(
        deciles_24[['decil_local', 'pct_ingreso']].rename(columns={'pct_ingreso': 'T4 2024'}),
        deciles_25[['decil_local', 'pct_ingreso']].rename(columns={'pct_ingreso': 'T4 2025'}),
        on='decil_local'
    )
    df_deciles_comp['Decil'] = "Decil " + df_deciles_comp['decil_local'].astype(str)

    fig_bar = px.bar(
        df_deciles_comp,
        x='Decil',
        y=['T4 2024', 'T4 2025'],
        barmode='group',
        labels={'value': 'Porcentaje del Ingreso Total (%)', 'variable': 'Período'},
        title="Participación de cada Decil en la Masa Total de Ingresos (Corrientes)"
    )
    st.plotly_chart(fig_bar, use_container_width=True)

    st.download_button(
        "⬇️ Descargar tabla de deciles (T4 2025) en CSV",
        data=deciles_25.to_csv(index=False).encode('utf-8'),
        file_name="deciles_locales_t4_2025.csv",
        mime="text/csv"
    )

    st.markdown("#### Comparación: Decil Local (Corrientes) vs. Decil Nacional (INDEC)")
    if col_decil_nac is None:
        st.info("La base cargada no trae la columna de decil nacional (`DECCFR` o `P_DECCF`), "
                "por lo que no es posible contrastar la posición local contra la nacional.")
    elif comparacion_nac_local_25.empty:
        st.info("No hay suficientes casos válidos para construir esta comparación.")
    else:
        st.caption(
            "Para cada decil local de Corrientes, se muestra el promedio del decil nacional "
            f"(`{col_decil_nac}`) de los hogares que lo integran."
        )
        st.dataframe(
            comparacion_nac_local_25.rename(columns={
                'decil_local': 'Decil Local',
                'decil_nacional_promedio': 'Decil Nacional Promedio',
                'n_casos': 'N° de casos',
                'brecha_local_vs_nacional': 'Brecha (Local - Nacional)'
            }).style.format({
                'Decil Nacional Promedio': '{:.2f}',
                'Brecha (Local - Nacional)': '{:+.2f}'
            }),
            use_container_width=True
        )

with tab3:
    st.subheader("Métricas de Brechas y Polarización de Ingresos")

    if not deciles_25.empty:
        d10_val = deciles_25.loc[deciles_25['decil_local'] == 10, 'pct_ingreso'].values[0]
        d1_val = deciles_25.loc[deciles_25['decil_local'] == 1, 'pct_ingreso'].values[0]
        brecha_d10_d1 = d10_val / d1_val if d1_val > 0 else 0

        st.write(f"**Brecha de Extremos (Decil 10 / Decil 1):** {brecha_d10_d1:.2f} veces.")
        st.write(f"**El 10% más rico (Decil 10) acumula el:** {d10_val:.2f}% de los ingresos.")
        st.write(f"**El 10% más pobre (Decil 1) acumula el:** {d1_val:.2f}% de los ingresos.")
        if palma_ratio is not None:
            st.write(f"**Ratio de Palma (D10 / D1-D4):** {palma_ratio:.2f}x.")

        resumen_export = pd.DataFrame([{
            'Gini_2024': gini_24,
            'Gini_2025': gini_25,
            'Diferencia_Gini': diff_gini,
            'Brecha_D10_D1': brecha_d10_d1,
            'Ratio_Palma': palma_ratio,
        }])
        st.download_button(
            "⬇️ Descargar resumen de indicadores en CSV",
            data=resumen_export.to_csv(index=False).encode('utf-8'),
            file_name="resumen_indicadores_desigualdad.csv",
            mime="text/csv"
        )

with tab4:
    st.subheader("¿De qué viven los hogares más vulnerables?")
    st.caption("Composición del ingreso por decil local: laboral vs. jubilaciones, indemnizaciones o subsidios/ayuda social.")

    columnas_todas_faltantes = set(faltantes_comp_24) | set(faltantes_comp_25)
    if columnas_todas_faltantes:
        st.warning(
            "La base cargada no incluye estas columnas de ingreso no laboral, por lo que se "
            f"completaron como 0 en el cálculo: {', '.join(sorted(columnas_todas_faltantes))}."
        )

    if composicion_25.empty:
        st.info("No hay datos suficientes para calcular la composición del ingreso.")
    else:
        composicion_25['Decil'] = "Decil " + composicion_25['decil_local'].astype(str)

        fig_comp = px.bar(
            composicion_25,
            x='Decil',
            y=['laboral_pct', 'jubilacion_pct', 'indemnizacion_pct', 'subsidios_pct'],
            barmode='stack',
            labels={'value': 'Porcentaje del ingreso del decil (%)', 'variable': 'Fuente de ingreso'},
            title="Composición del Ingreso por Decil Local — Corrientes, T4 2025"
        )
        nombres_fuente = {
            'laboral_pct': 'Laboral',
            'jubilacion_pct': 'Jubilaciones/Pensiones',
            'indemnizacion_pct': 'Indemnizaciones',
            'subsidios_pct': 'Subsidios/Ayuda social'
        }
        fig_comp.for_each_trace(lambda t: t.update(name=nombres_fuente.get(t.name, t.name)))
        st.plotly_chart(fig_comp, use_container_width=True)

        st.markdown("#### Foco en los deciles más vulnerables (D1 a D3)")
        vulnerables = composicion_25[composicion_25['decil_local'].isin([1, 2, 3])]
        if not vulnerables.empty:
            c1, c2, c3 = st.columns(3)
            c1.metric("D1 - % No Laboral", f"{vulnerables.loc[vulnerables['decil_local']==1, 'no_laboral_pct'].values[0]:.1f}%")
            c2.metric("D2 - % No Laboral", f"{vulnerables.loc[vulnerables['decil_local']==2, 'no_laboral_pct'].values[0]:.1f}%")
            c3.metric("D3 - % No Laboral", f"{vulnerables.loc[vulnerables['decil_local']==3, 'no_laboral_pct'].values[0]:.1f}%")

            comp_top = composicion_25.loc[composicion_25['decil_local'] == 10, 'no_laboral_pct']
            if not comp_top.empty:
                st.write(
                    f"En contraste, el Decil 10 depende de fuentes no laborales en apenas "
                    f"**{comp_top.values[0]:.1f}%** de su ingreso, frente a un promedio de "
                    f"**{vulnerables['no_laboral_pct'].mean():.1f}%** en los Deciles 1 a 3."
                )

        st.dataframe(
            composicion_25[['Decil', 'laboral_pct', 'jubilacion_pct', 'indemnizacion_pct', 'subsidios_pct', 'no_laboral_pct']]
            .rename(columns={
                'laboral_pct': 'Laboral (%)',
                'jubilacion_pct': 'Jubilaciones (%)',
                'indemnizacion_pct': 'Indemnizaciones (%)',
                'subsidios_pct': 'Subsidios (%)',
                'no_laboral_pct': 'Total No Laboral (%)'
            })
            .style.format({
                'Laboral (%)': '{:.1f}', 'Jubilaciones (%)': '{:.1f}',
                'Indemnizaciones (%)': '{:.1f}', 'Subsidios (%)': '{:.1f}',
                'Total No Laboral (%)': '{:.1f}'
            }),
            use_container_width=True
        )

        st.download_button(
            "⬇️ Descargar composición del ingreso por decil (CSV)",
            data=composicion_25.to_csv(index=False).encode('utf-8'),
            file_name="composicion_ingreso_por_decil_t4_2025.csv",
            mime="text/csv"
        )

Overwriting app.py


In [5]:
# 3) Levantar Streamlit en segundo plano
import subprocess, time, socket

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

log_file = open("logs_streamlit.txt", "w")
proceso_streamlit = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT
)

puerto_listo = False
for _ in range(30):
    time.sleep(1)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("localhost", 8501)) == 0:
            puerto_listo = True
            break

if puerto_listo:
    print("✅ Streamlit está corriendo en el puerto 8501.")
else:
    print("⚠️ No respondió a tiempo. Log:")
    print(open("logs_streamlit.txt").read()[-2000:])

✅ Streamlit está corriendo en el puerto 8501.


In [ ]:
# 4) Túnel público - localtunnel (gratis, sin cuenta)
!npx --yes localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://long-games-sort.loca.lt


In [ ]:
!wget -q -O - https://loca.lt/mytunnelpassword

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("TU_TOKEN_AQUI")  # desde https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.kill()
public_url = ngrok.connect(8501, "http")
print("Dashboard disponible en:", public_url)